# Tutorial of GlimpsePrune (LLaVA1.5)

In [ ]:
import os

import sys
parent_path = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if parent_path not in sys.path:
    sys.path.append(parent_path)

import torch
from llava.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN
from llava.conversation import conv_templates
from llava_gp.mm_utils import (
    get_model_name_from_path,
    process_images,
    process_bboxes,
    tokenizer_image_token,
)
from llava_gp.model.builder import load_pretrained_model
from transformers import GenerationConfig
from PIL import Image
import matplotlib.pyplot as plt


In [ ]:
def apply_mask_on_image(image: Image.Image, 
                        mask: torch.Tensor, 
                        alpha: float=0.4, 
                        color: tuple=(0, 255, 0),
                        patch_size: int=28) -> Image.Image:
    """
    Apply a boolean mask on the image.
    Args:
        image (Image.Image): The input image.
        mask (torch.Tensor): A boolean mask of shape (H, W).
    Returns:
        Image.Image: The blended image with the mask applied.
    """
    mask_h, mask_w = mask.shape
    target_h = mask_h * patch_size
    target_w = mask_w * patch_size
    # resize image and mask
    image = image.resize((target_w, target_h), Image.Resampling.LANCZOS)
    mask = torch.nn.functional.interpolate(
        mask.unsqueeze(0).unsqueeze(0).float(),
        size=(target_h, target_w),
        mode='nearest'
    ).squeeze(0).squeeze(0).bool()

    # apply mask with color and alpha
    mask_image = Image.new('RGBA', image.size, color + (0,))
    alpha_data = (mask * 255 * alpha).byte().cpu().numpy()
    alpha_channel = Image.fromarray(alpha_data, mode='L')
    mask_image.putalpha(alpha_channel)
    blended_image = Image.alpha_composite(image.convert('RGBA'), mask_image)
    return blended_image.convert('RGB')
    

In [ ]:
base_model = "liuhaotian/llava-v1.5-13b"
new_modules_dir = "ashun989/GlimpsePrune_LLaVA-1.5-13B"
show_image_token_masks = True
max_remain_ratio = 0.111
device = "cuda:0"
torch_dtype = torch.bfloat16

model_name = get_model_name_from_path(base_model)
llava_model_args = {
    "attn_implementation": "flash_attention_2",
    "torch_dtype": torch_dtype,
}

tokenizer, model, image_processor, max_length = load_pretrained_model(
    base_model, None, model_name, device_map=device, **llava_model_args)

model.load_new_modules(new_modules_dir)
model.eval()
conv_mode = "vicuna_v1"


In [ ]:
if max_remain_ratio is not None:
    model.config.max_remain_ratio = max_remain_ratio

generation_config = GenerationConfig.from_pretrained(
    base_model,
)
generation_config.max_new_tokens = 1024
print("Generation config:", generation_config)

## Single Image Inference

In [ ]:
query = "What kind of a tie is the groom wearing?"
img_path = "../examples/people.png"

input_ids = []
images = []
image_sizes = []
grid_h = grid_w = model.get_vision_tower().num_patches_per_side
if model.config.mm_use_im_start_end:
    query = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN + '\n' + query
else:
    query = DEFAULT_IMAGE_TOKEN + '\n' + query
conv = conv_templates[conv_mode].copy()
conv.append_message(conv.roles[0], query)
conv.append_message(conv.roles[1], None)
prompt = conv.get_prompt()

image = Image.open(img_path).convert("RGB")
image_tensor = process_images([image], image_processor, model.config)[0]
input_id = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt')
input_ids.append(input_id)
images.append(image_tensor)
image_sizes.append(image.size)

input_ids = torch.stack(input_ids, dim=0).to(device=device)
images = torch.stack(images, dim=0).to(device=device, dtype=torch_dtype)

generate_ids = model.generate(
            input_ids=input_ids,
            images=images,
            image_sizes=image_sizes,
            generation_config=generation_config,
            do_selection=True,  # Enable glimpse prune
        )
response = tokenizer.batch_decode(generate_ids, skip_special_tokens=True,
                                  clean_up_tokenization_spaces=True)[0].strip()
print(response)

torch.cuda.empty_cache()

In [ ]:
if show_image_token_masks:
    attention_mask = torch.ones(input_ids.shape, device=device, dtype=torch.long)
    with torch.inference_mode():
        image_token_bool_mask = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            images=images,
            image_sizes=image_sizes,
            return_dict=True,
            do_selection=True).image_token_bool_masks[0]
    image_token_bool_mask = image_token_bool_mask.reshape(grid_h, grid_w)
        
    retain_ratio = image_token_bool_mask.float().mean().item()
    print(f"Retain ratio: {retain_ratio*100:.1f}%")

    blended_image = apply_mask_on_image(
        image=Image.open(img_path),
        mask=image_token_bool_mask,
        patch_size=grid_h
    )

    # Display the blended image
    plt.figure(figsize=(8, 8))
    plt.imshow(blended_image)
    plt.axis("off")
    plt.show()

    torch.cuda.empty_cache()